# TP L23 — Classification rare sans fuite de test

**Public :** première pratique du ML supervisé · **Durée :** 60 minutes

Ce TP sépare entraînement, validation et test scellé. Le seuil est choisi sur la validation selon le coût des erreurs ; le test final n'est ouvert qu'une fois.

## 1. Le contrat des trois jeux

- **entraînement** : ajuster le score ;
- **validation** : choisir le seuil et les hyperparamètres ;
- **test scellé** : produire une seule mesure finale.

Réutiliser le test pour régler le seuil crée une fuite et transforme le résultat final en nouvelle validation.

In [ ]:
from pathlib import Path
import sys

LAB_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'rare_classification_lab.py').is_file())
if str(LAB_ROOT) not in sys.path:
    sys.path.insert(0, str(LAB_ROOT))

from rare_classification_lab import (
    classification_metrics, confusion_counts, fit_score_model,
    generate_split, run_experiment, select_threshold,
)

## 2. Construire des jeux disjoints

Les identifiants rendent le non-chevauchement vérifiable. Les données sont synthétiques : elles illustrent un contrat expérimental, pas un taux d'incident réel.

In [ ]:
train = generate_split('train', seed=101, size=2000)
validation = generate_split('validation', seed=211, size=1200)
test_scelle = generate_split('test', seed=307, size=1200)

for nom, jeu in [('train', train), ('validation', validation), ('test', test_scelle)]:
    prevalence = sum(item.incident for item in jeu) / len(jeu)
    print(nom, len(jeu), f'{prevalence:.2%} incidents')

assert {x.sample_id for x in train}.isdisjoint(x.sample_id for x in validation)
assert {x.sample_id for x in train}.isdisjoint(x.sample_id for x in test_scelle)

## 3. Apprendre le score, puis choisir le seuil

Le modèle voit seulement l'entraînement. Le seuil voit seulement la validation. Une fausse négative coûte ici douze fois une fausse alerte : ce nombre est une hypothèse explicite à soumettre à l'opérateur.

In [ ]:
modele = fit_score_model(train)
choix = select_threshold(validation, modele, false_positive_cost=1.0, false_negative_cost=12.0)
print(choix)
assert choix['threshold'] != 0.5

## 4. Ouvrir le test une fois

Après cette cellule, le résultat est final. Si nous changeons le modèle ou le seuil en le regardant, ce jeu cesse d'être un test. Le champ `test_open_count = 1` est une assertion de protocole, pas un compteur instrumenté. Les empreintes identifient les partitions ; elles ne prouvent ni l'ordre des accès ni l'absence de consultations lors d'exécutions antérieures.

In [ ]:
experience = run_experiment()
final = experience['final_test']
reference = experience['naive_threshold_reference']
print('seuil validé', final)
print('seuil 0,5', reference)
assert experience['split_contract']['test_open_count'] == 1
assert final['threshold'] == choix['threshold']

## 5. Interpréter sans masquer la classe rare

L'exactitude peut rester élevée en prédisant presque toujours « normal ». Il faut donc publier VP, FP, VN, FN, précision, rappel, F1 et le coût explicite, pas un score unique.

## 6. À vous

1. Faites passer le coût d'une omission de 12 à 4. Comment le seuil change-t-il ?
2. Mesurez le nombre d'alertes supplémentaires demandé par le seuil plus sensible.
3. Expliquez pourquoi choisir le meilleur seuil sur `test_scelle` serait une fuite.
4. Ajoutez une contrainte de capacité : au plus 30 alertes examinées par période.

In [ ]:
comptes = confusion_counts(test_scelle, modele, choix['threshold'])
metriques = classification_metrics(comptes)
assert sum(comptes.values()) == len(test_scelle)
assert 0.0 <= metriques['precision'] <= 1.0
assert 0.0 <= metriques['recall'] <= 1.0
print('Contrat train/validation/test vérifié.')